In [1]:
import pandas as pd

In [2]:
policies = pd.read_csv('policies_bias2_01_09.csv')
#policies2 = pd.read_csv('policies2_bias2.csv')

#policies = pd.concat([policies1, policies2])
policies = policies[['sample_size',
         'ranking_method', 'f1_test', 'roc_test_ovo', 'roc_test_ovr']]

for column in ['f1_test', 'roc_test_ovo', 'roc_test_ovr']:
    policies[column] = policies[column]*100

grouped_df = (policies.groupby(['sample_size', 'ranking_method'])
                     .agg(['mean', 'min', 'max'])
                     .reset_index()
                     .round(1)
             )
print(grouped_df.to_latex())

\begin{tabular}{lrlrrrrrrrrr}
\toprule
{} & sample\_size &          ranking\_method & \multicolumn{3}{l}{f1\_test} & \multicolumn{3}{l}{roc\_test\_ovo} & \multicolumn{3}{l}{roc\_test\_ovr} \\
{} &    mean &   min &   max &         mean &   min &   max &         mean &   min &   max \\
\midrule
0 &       40000 &                     CCR &    57.5 &  52.2 &  61.4 &         88.9 &  85.1 &  90.4 &         89.9 &  86.6 &  91.2 \\
1 &       40000 &           max\_confusion &    58.1 &  52.7 &  61.2 &         88.8 &  86.3 &  91.0 &         89.8 &  87.4 &  91.7 \\
2 &       40000 &  max\_pairwise\_confusion &    58.1 &  51.2 &  62.9 &         88.8 &  86.4 &  90.3 &         89.7 &  87.6 &  91.0 \\
3 &       40000 &             no\_priority &    59.0 &  54.5 &  62.3 &         89.2 &  88.0 &  90.4 &         90.0 &  89.1 &  91.1 \\
4 &       40000 &              proportion &    57.7 &  49.3 &  62.0 &         88.3 &  82.9 &  90.1 &         89.3 &  85.0 &  90.8 \\
\bottomrule
\end{tabular}



In [3]:
columns_list = policies.columns.tolist()
print(columns_list)

['sample_size', 'ranking_method', 'f1_test', 'roc_test_ovo', 'roc_test_ovr']


In [4]:
sn = pd.read_csv('results_snr_bias1_032025.csv')

sn = sn[['sn_ratio', 'f1_test', 'roc_test_ovo', 'roc_test_ovr', 'opt_method']]

for column in ['f1_test', 'roc_test_ovo', 'roc_test_ovr']:
    sn[column] = sn[column]*100

grouped_df = (sn.groupby(['opt_method', 'sn_ratio'])
                     .agg(['mean', 'min', 'max'])
                     .reset_index()
                     .round(1)
             )
print(grouped_df.to_latex())

\begin{tabular}{llrrrrrrrrrr}
\toprule
{} & opt\_method & sn\_ratio & \multicolumn{3}{l}{f1\_test} & \multicolumn{3}{l}{roc\_test\_ovo} & \multicolumn{3}{l}{roc\_test\_ovr} \\
{} &    mean &   min &   max &         mean &   min &   max &         mean &   min &   max \\
\midrule
0 &    oneloss &        4 &    59.7 &  51.3 &  64.3 &         90.0 &  86.6 &  91.8 &         89.3 &  86.0 &  90.8 \\
1 &    oneloss &        6 &    60.7 &  52.3 &  66.1 &         90.4 &  87.7 &  91.6 &         89.3 &  87.3 &  90.8 \\
2 &    oneloss &        8 &    60.4 &  48.6 &  66.7 &         90.4 &  87.1 &  92.1 &         89.4 &  85.5 &  91.0 \\
3 &    oneloss &       10 &    59.9 &  51.0 &  65.0 &         90.3 &  87.3 &  91.6 &         89.2 &  86.1 &  90.6 \\
4 &  twolosses &        4 &    61.2 &  55.6 &  67.0 &         90.9 &  88.8 &  92.3 &         90.2 &  87.6 &  92.1 \\
5 &  twolosses &        6 &    60.8 &  55.7 &  67.0 &         90.6 &  87.2 &  91.7 &         90.1 &  87.7 &  91.3 \\
6 &  twolosses &   

In [5]:
from scipy.stats import mannwhitneyu, ttest_ind, ttest_rel, wilcoxon
sn = pd.read_csv('results_snr_bias1_032025.csv')


sn = sn[['sn_ratio', 'f1_test', 'roc_test_ovo', 'roc_test_ovr', 'opt_method']]

def apply_test(value=4, metric='f1_test', value_colum = 'sn_ratio', significance=0.05):

    s1 = sn[(sn[value_colum]==value) & (sn['opt_method']=='oneloss')][metric]
    
    s2 = sn[(sn[value_colum]==value) & (sn['opt_method']=='twolosses')][metric]
    
    _, p = mannwhitneyu(s1, s2)

    print(value, metric, p<significance, p, 'mannwhitneyu')

    _, p = ttest_ind(s1, s2, alternative="less")

    print(value, metric, p<=significance, p, 'ttest')

    _, p = ttest_rel(s1, s2, alternative="less")

    print(value, metric, p<=significance, p, 'paired ttest')

    _, p = wilcoxon(s1, s2, alternative="less")

    print(value, metric, p<=significance, p, 'wilcoxon')


for v in [4, 6, 8, 10]: 
    for m in ['f1_test', 'roc_test_ovo', 'roc_test_ovr']:   
            apply_test(value=v, metric=m)

4 f1_test False 0.054344886607223704 mannwhitneyu
4 f1_test False 0.05085329085956825 ttest
4 f1_test True 0.022881124205909462 paired ttest
4 f1_test True 0.031341406249789885 wilcoxon
4 roc_test_ovo True 0.003486220747285127 mannwhitneyu
4 roc_test_ovo True 0.001014074857448411 ttest
4 roc_test_ovo True 0.0002818950899134801 paired ttest
4 roc_test_ovo True 0.0004472150323730773 wilcoxon
4 roc_test_ovr True 0.00047603701963630924 mannwhitneyu
4 roc_test_ovr True 0.0002711027647433448 ttest
4 roc_test_ovr True 7.680296277831419e-05 paired ttest
4 roc_test_ovr True 8.711406679827146e-05 wilcoxon
6 f1_test False 0.3530857441406148 mannwhitneyu
6 f1_test False 0.4547771603467563 ttest
6 f1_test False 0.45676348808131023 paired ttest
6 f1_test False 0.5531778064366975 wilcoxon
6 roc_test_ovo False 0.12290690122947068 mannwhitneyu
6 roc_test_ovo False 0.17689414111078539 ttest
6 roc_test_ovo False 0.15685781231960821 paired ttest
6 roc_test_ovo False 0.10294411152803701 wilcoxon
6 roc_test

In [6]:
import pandas as pd 

sn = pd.read_csv('results_snr_bias2_03042025.csv')

sn = sn[['sn_ratio', 'f1_test', 'roc_test_ovo', 'roc_test_ovr', 'opt_method']]

for column in ['f1_test', 'roc_test_ovo', 'roc_test_ovr']:
    sn[column] = sn[column]*100

grouped_df = (sn.groupby(['opt_method', 'sn_ratio'])
                     .agg(['mean', 'min', 'max'])
                     .reset_index()
                     .round(1)
             )
print(grouped_df.to_latex())

\begin{tabular}{llrrrrrrrrrr}
\toprule
{} & opt\_method & sn\_ratio & \multicolumn{3}{l}{f1\_test} & \multicolumn{3}{l}{roc\_test\_ovo} & \multicolumn{3}{l}{roc\_test\_ovr} \\
{} &    mean &   min &   max &         mean &   min &   max &         mean &   min &   max \\
\midrule
0 &    oneloss &        4 &    61.3 &  54.1 &  65.3 &         90.1 &  88.1 &  91.7 &         90.8 &  88.5 &  92.4 \\
1 &    oneloss &        6 &    61.1 &  55.2 &  68.0 &         90.1 &  86.9 &  92.4 &         90.8 &  87.9 &  92.7 \\
2 &    oneloss &        8 &    60.5 &  49.6 &  66.4 &         90.0 &  84.6 &  91.8 &         90.7 &  85.4 &  92.4 \\
3 &    oneloss &       10 &    60.2 &  51.6 &  65.3 &         89.9 &  86.6 &  91.2 &         90.6 &  87.9 &  91.9 \\
4 &  twolosses &        4 &    61.5 &  36.8 &  67.4 &         90.3 &  78.6 &  92.3 &         91.4 &  80.5 &  93.2 \\
5 &  twolosses &        6 &    61.7 &  45.1 &  68.0 &         90.6 &  82.6 &  92.2 &         91.6 &  83.8 &  93.1 \\
6 &  twolosses &   

In [7]:
sn = pd.read_csv('results_snr_bias2_03042025.csv')

sn = sn[['sn_ratio', 'f1_test', 'roc_test_ovo', 'roc_test_ovr', 'opt_method']]

for v in [4, 6, 8, 10]: 
    for m in ['f1_test', 'roc_test_ovo', 'roc_test_ovr']:   
            apply_test(value=v, metric=m)

4 f1_test False 0.12902574754311813 mannwhitneyu
4 f1_test False 0.4302392207754736 ttest
4 f1_test False 0.4269816536786894 paired ttest
4 f1_test False 0.2024173610787104 wilcoxon
4 roc_test_ovo True 0.006105969832601182 mannwhitneyu
4 roc_test_ovo False 0.3281761385043078 ttest
4 roc_test_ovo False 0.321869335515771 paired ttest
4 roc_test_ovo False 0.05319708659530697 wilcoxon
4 roc_test_ovr True 0.0001885197820415843 mannwhitneyu
4 roc_test_ovr False 0.0970701459515698 ttest
4 roc_test_ovr False 0.08933180079407833 paired ttest
4 roc_test_ovr True 0.004108368228795213 wilcoxon
6 f1_test False 0.06672704379389652 mannwhitneyu
6 f1_test False 0.27965259367270395 ttest
6 f1_test False 0.2484110003387725 paired ttest
6 f1_test False 0.13557759885441611 wilcoxon
6 roc_test_ovo True 0.002914084219557631 mannwhitneyu
6 roc_test_ovo False 0.09770862857737649 ttest
6 roc_test_ovo False 0.07329931978227255 paired ttest
6 roc_test_ovo True 0.008282763489715197 wilcoxon
6 roc_test_ovr True 0.

In [8]:
import pandas as pd 

sn = pd.read_csv('results_lenght_bias1.csv')

sn = sn[['seq_length', 'f1_test', 'roc_test_ovo', 'roc_test_ovr', 'opt_method']]

for column in ['f1_test', 'roc_test_ovo', 'roc_test_ovr']:
    sn[column] = sn[column]*100

grouped_df = (sn.groupby(['opt_method', 'seq_length'])
                     .agg(['mean', 'min', 'max'])
                     .reset_index()
                     .round(1)
             )
print(grouped_df.to_latex())

\begin{tabular}{llrrrrrrrrrr}
\toprule
{} & opt\_method & seq\_length & \multicolumn{3}{l}{f1\_test} & \multicolumn{3}{l}{roc\_test\_ovo} & \multicolumn{3}{l}{roc\_test\_ovr} \\
{} &    mean &   min &   max &         mean &   min &   max &         mean &   min &   max \\
\midrule
0 &    oneloss &         50 &    46.4 &  38.1 &  53.2 &         84.7 &  79.7 &  86.3 &         82.2 &  76.5 &  84.2 \\
1 &    oneloss &        150 &    57.4 &  51.1 &  61.6 &         88.8 &  87.3 &  90.6 &         87.5 &  85.5 &  89.2 \\
2 &  twolosses &         50 &    49.3 &  43.8 &  55.5 &         85.6 &  81.3 &  87.2 &         83.3 &  78.4 &  85.2 \\
3 &  twolosses &        150 &    57.0 &  51.7 &  60.7 &         88.8 &  85.6 &  90.0 &         87.3 &  85.1 &  88.4 \\
\bottomrule
\end{tabular}



In [9]:
sn = pd.read_csv('results_lenght_bias1.csv')

sn = sn[['seq_length', 'f1_test', 'roc_test_ovo', 'roc_test_ovr', 'opt_method']]

for v in [50, 150]: 
    for m in ['f1_test', 'roc_test_ovo', 'roc_test_ovr']:   
            apply_test(value=v, metric=m, value_colum='seq_length')

50 f1_test True 0.003979497830522786 mannwhitneyu
50 f1_test True 0.0015816797345647006 ttest
50 f1_test True 0.0028565260182585642 paired ttest
50 f1_test True 0.003417928212650613 wilcoxon
50 roc_test_ovo True 0.006105969832601182 mannwhitneyu
50 roc_test_ovo True 0.007809098956275832 ttest
50 roc_test_ovo True 0.009746292814411334 paired ttest
50 roc_test_ovo True 0.0018047166746921076 wilcoxon
50 roc_test_ovr True 0.002213596583586509 mannwhitneyu
50 roc_test_ovr True 0.0040624587468443335 ttest
50 roc_test_ovr True 0.005705431054757776 paired ttest
50 roc_test_ovr True 0.001580882468327745 wilcoxon
150 f1_test False 0.2896471052909906 mannwhitneyu
150 f1_test False 0.7556730556475145 ttest
150 f1_test False 0.7709129664904184 paired ttest
150 f1_test False 0.7000320359161272 wilcoxon
150 roc_test_ovo False 0.28461008187076475 mannwhitneyu
150 roc_test_ovo False 0.5952475229874287 ttest
150 roc_test_ovo False 0.6049867042571611 paired ttest
150 roc_test_ovo False 0.5286947437711744

In [10]:
import pandas as pd 

sn = pd.read_csv('results_lenght_bias2.csv')

sn = sn[['seq_length', 'f1_test', 'roc_test_ovo', 'roc_test_ovr', 'opt_method']]

for column in ['f1_test', 'roc_test_ovo', 'roc_test_ovr']:
    sn[column] = sn[column]*100

grouped_df = (sn.groupby(['opt_method', 'seq_length'])
                     .agg(['mean', 'min', 'max'])
                     .reset_index()
                     .round(1)
             )
print(grouped_df.to_latex())



\begin{tabular}{llrrrrrrrrrr}
\toprule
{} & opt\_method & seq\_length & \multicolumn{3}{l}{f1\_test} & \multicolumn{3}{l}{roc\_test\_ovo} & \multicolumn{3}{l}{roc\_test\_ovr} \\
{} &    mean &   min &   max &         mean &   min &   max &         mean &   min &   max \\
\midrule
0 &    oneloss &         50 &    52.9 &  48.0 &  56.5 &         85.9 &  83.8 &  87.1 &         87.2 &  85.2 &  88.4 \\
1 &    oneloss &        150 &    58.3 &  51.6 &  63.0 &         88.6 &  86.7 &  90.0 &         89.5 &  87.4 &  90.7 \\
2 &  twolosses &         50 &    53.2 &  49.3 &  57.3 &         85.6 &  82.1 &  87.4 &         86.9 &  83.8 &  88.5 \\
3 &  twolosses &        150 &    58.0 &  49.7 &  62.8 &         88.5 &  86.1 &  90.1 &         89.5 &  87.2 &  90.9 \\
\bottomrule
\end{tabular}



In [11]:
sn = pd.read_csv('results_lenght_bias2.csv')

sn = sn[['seq_length', 'f1_test', 'roc_test_ovo', 'roc_test_ovr', 'opt_method']]

for v in [50, 150]: 
    for m in ['f1_test', 'roc_test_ovo', 'roc_test_ovr']:   
            apply_test(value=v, metric=m, value_colum='seq_length')

50 f1_test False 0.41512764195559815 mannwhitneyu
50 f1_test False 0.30695835615861605 ttest
50 f1_test False 0.3282537042320355 paired ttest
50 f1_test False 0.35944378106813346 wilcoxon
50 roc_test_ovo False 0.12902574754311813 mannwhitneyu
50 roc_test_ovo False 0.906807985071383 ttest
50 roc_test_ovo False 0.8941268257600311 paired ttest
50 roc_test_ovo False 0.9205722503574022 wilcoxon
50 roc_test_ovr False 0.11992499559553982 mannwhitneyu
50 roc_test_ovr False 0.8951282018108702 ttest
50 roc_test_ovr False 0.8883477887551923 paired ttest
50 roc_test_ovr False 0.900694895027227 wilcoxon
150 f1_test False 0.358594406803188 mannwhitneyu
150 f1_test False 0.6520980538690446 ttest
150 f1_test False 0.6404152501066855 paired ttest
150 f1_test False 0.35178184993500383 wilcoxon
150 roc_test_ovo False 0.3696994096557248 mannwhitneyu
150 roc_test_ovo False 0.6929541858204977 ttest
150 roc_test_ovo False 0.6809676923165294 paired ttest
150 roc_test_ovo False 0.5450346159206327 wilcoxon
150 

In [12]:
loss_function = pd.read_csv('loss_function.csv')

loss_function = loss_function[['loss', 'f1_test', 'roc_test_ovo', 'roc_test_ovr']]

for column in ['f1_test', 'roc_test_ovo', 'roc_test_ovr']:
    loss_function[column] = loss_function[column]*100

grouped_df = (loss_function.groupby(['loss'])
                     .agg(['mean', 'min', 'max'])
                     .reset_index()
                     .round(1)
             )
print(grouped_df.to_latex())

\begin{tabular}{llrrrrrrrrr}
\toprule
{} &                           loss & \multicolumn{3}{l}{f1\_test} & \multicolumn{3}{l}{roc\_test\_ovo} & \multicolumn{3}{l}{roc\_test\_ovr} \\
{} &    mean &   min &   max &         mean &   min &   max &         mean &   min &   max \\
\midrule
0 &               CrossEntropyLoss &    57.6 &  55.4 &  62.4 &         88.5 &  86.8 &  89.7 &         87.5 &  85.7 &  89.0 \\
1 &                        NLLLoss &    56.5 &  54.1 &  59.7 &         87.9 &  86.9 &  89.2 &         87.0 &  85.9 &  88.0 \\
2 &                      focalLoss &    56.7 &  52.4 &  60.0 &         88.5 &  86.9 &  89.6 &         87.4 &  85.9 &  88.5 \\
3 &  non\_weighted\_CrossEntropyLoss &    55.0 &  48.9 &  60.5 &         88.7 &  85.5 &  91.0 &         87.7 &  84.0 &  90.1 \\
\bottomrule
\end{tabular}



In [13]:
import pandas as pd
#complete_instance.csv bias 1
# complete_bias2.csv bias 2
complete_instance = pd.read_csv('complete_bias2.csv')

complete_instance = complete_instance[['opt_method', 'f1_test', 'roc_test_ovo', 'roc_test_ovr']]

for column in ['f1_test', 'roc_test_ovo', 'roc_test_ovr']:
    complete_instance[column] = complete_instance[column]*100

grouped_df = (complete_instance.groupby(['opt_method'])
                     .agg(['mean',  'min', 'max'])
                     .reset_index()
                     .round(2)
             )
print(grouped_df.to_latex())

\begin{tabular}{llrrrrrrrrr}
\toprule
{} & opt\_method & \multicolumn{3}{l}{f1\_test} & \multicolumn{3}{l}{roc\_test\_ovo} & \multicolumn{3}{l}{roc\_test\_ovr} \\
{} &    mean &    min &    max &         mean &    min &    max &         mean &    min &    max \\
\midrule
0 &    oneloss &   67.09 &  57.60 &  71.48 &        92.93 &  90.77 &  94.17 &        93.41 &  91.62 &  94.52 \\
1 &  twolosses &   66.28 &  60.49 &  71.37 &        92.53 &  89.58 &  94.01 &        93.05 &  90.20 &  94.39 \\
\bottomrule
\end{tabular}

